In [1]:
# automatically reload imported modules before executing code

%load_ext autoreload
%autoreload 2

In [48]:
from pyrekordbox import Rekordbox6Database
import polars as pl
from nbutils import setup_path

setup_path()
db = Rekordbox6Database()

pl.Config.set_tbl_rows(30)  # Show 100 row

[22:16:20] pyrekordbox.db6.database:WARNING  - Rekordbox is running!


polars.config.Config

In [3]:
from utils import get_base_dataset

base_df = get_base_dataset(db, min_tag_count=5)


Filtering tags with fewer than 5 occurrences:

Genre:
  - Grime: 1 occurrence(s)
  - Dancehall: 3 occurrence(s)
  - New Beat: 3 occurrence(s)
  - Gabber: 4 occurrence(s)
  - Blues: 4 occurrence(s)

Mood:
  - Industrial: 1 occurrence(s)

Total tags filtered: 6



In [ ]:
from processing import preprocess_tag_group


# Load features
features_df = pl.read_parquet("../data/song_features.parquet")

# Define feature columns to exclude
exclude_cols = [
    "song_path",
    "harmonic_percussive_ratio",
    "percussive_strength",
    "tonnetz_mean_0", "tonnetz_mean_1", "tonnetz_mean_2", "tonnetz_mean_3", "tonnetz_mean_4", "tonnetz_mean_5",
    "tonnetz_std_0", "tonnetz_std_1", "tonnetz_std_2", "tonnetz_std_3", "tonnetz_std_4", "tonnetz_std_5"
]

# Get feature columns
feature_cols = [col for col in features_df.columns 
                if col not in exclude_cols + ["song_id", "song_path"]]

# Filter features to only include non-null energy_increase_ratio
features_df = features_df.filter(pl.col("energy_increase_ratio").is_not_null())

# Configuration
test_size = 0.2
random_state = 42
min_train_count = 10
pca_variance = 0.95


# MOOD
mood = preprocess_tag_group(
    base_df, features_df, "Mood",
    feature_cols=feature_cols,
    test_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=True,
    pca_variance=0.95
)

# SITUATION
situation = preprocess_tag_group(
    base_df, features_df, "Situation",
    feature_cols=feature_cols,
    test_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=True,
    pca_variance=0.95
)

# Much cleaner - use named attributes!
genre = preprocess_tag_group(
    base_df, features_df, "Genre",
    feature_cols=feature_cols,
    test_size=0.2,
    min_train_count=10,
    apply_scaling=True,
    apply_pca=False,
    pca_variance=0.95
)


print("\n" + "="*70)
print("PREPROCESSING COMPLETE")
print("="*70)
print(f"Ready to train models for {len(mood.tags) + len(situation.tags) + len(genre.tags)} total labels")
print("="*70 + "\n")


PREPROCESSING: MOOD

Preparing multi-label data for 'Mood':
Unique songs: 686
Unique tags: 23
Tags: Chill, Dancefloor, Dark, Deep, Experimental, Good Vibes, Groovy, Guilty, Hippy, Like a boss, Love, Minimal, Mysterious, Pretty, Sad, Spacy, Stadium, Sunset, Synths, Temposhifter, Trippy, Uplifting, Uptempo

Final dataset shape:
  X: (496, 361) (song_id + 360 features)
  y: (496, 23) (23 binary labels)
  Average tags per song: 4.33


Mood - Initial label statistics:
shape: (23, 3)
┌──────────────┬───────┬────────────┐
│ tag          ┆ count ┆ percentage │
│ ---          ┆ ---   ┆ ---        │
│ str          ┆ i64   ┆ f64        │
╞══════════════╪═══════╪════════════╡
│ Good Vibes   ┆ 281   ┆ 56.653226  │
│ Dancefloor   ┆ 278   ┆ 56.048387  │
│ Uplifting    ┆ 259   ┆ 52.217742  │
│ Sunset       ┆ 231   ┆ 46.572581  │
│ Groovy       ┆ 211   ┆ 42.540323  │
│ Deep         ┆ 133   ┆ 26.814516  │
│ Chill        ┆ 123   ┆ 24.798387  │
│ Pretty       ┆ 112   ┆ 22.580645  │
│ Minimal      ┆ 79   

In [102]:
from models import (
    get_linear_model,
    get_knn_model,
    get_decision_tree_model,
    get_random_forest_model,
    get_xgboost_model,
    evaluate_model,
    train_and_compare_models,
    predictions_to_labels,
)

processing_result = genre

models = {
    "Linear": get_linear_model(),
    # "KNN": get_knn_model(n_neighbors=10),
    # "Decision Tree": get_decision_tree_model(max_depth=15),
    "Random Forest": get_random_forest_model(n_estimators=100),
    "XGBoost": get_xgboost_model(n_estimators=100),
}


# Train & compare globally
comparison = train_and_compare_models(
    models, 
    processing_result.X_train, processing_result.y_train,
    processing_result.X_test, processing_result.y_test,
    processing_result.tags
)

# detailed breakdown
for model_name, model in models.items():
    print( (15 * "*-") + f" Evaluating {model_name} " + (15 * "*-"))
    metrics = evaluate_model(
        model=model, 
        X_test=processing_result.X_test, 
        y_test=processing_result.y_test, 
        X_train=processing_result.X_train, # For threshold optim
        y_train=processing_result.y_train, # For threshold optim
        tags=processing_result.tags,
        optimize_thresholds=True
    )



Training: Linear
 Linear trained successfully
  Hamming Loss: 0.1347, Exact Match: 0.0300, Macro F1: 0.2978, Weighted F1: 0.4531

Training: Random Forest
 Random Forest trained successfully
  Hamming Loss: 0.0923, Exact Match: 0.0100, Macro F1: 0.1523, Weighted F1: 0.3502

Training: XGBoost
 XGBoost trained successfully
  Hamming Loss: 0.0913, Exact Match: 0.0500, Macro F1: 0.2075, Weighted F1: 0.3808

MODEL COMPARISON SUMMARY

Sorted by Macro F1-Score:
shape: (3, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ model     ┆ hamming_l ┆ exact_mat ┆ macro_pre ┆ … ┆ macro_f1 ┆ weighted_ ┆ weighted_ ┆ weighted_ │
│ ---       ┆ oss       ┆ ch_accura ┆ cision    ┆   ┆ ---      ┆ precision ┆ recall    ┆ f1        │
│ str       ┆ ---       ┆ cy        ┆ ---       ┆   ┆ f64      ┆ ---       ┆ ---       ┆ ---       │
│           ┆ f64       ┆ ---       ┆ f64       ┆   ┆          ┆ f64       ┆ f64       ┆ f64       │
│           ┆      

In [ ]:
from utils import get_clean_songs

songs_df = get_clean_songs(db, rename=True)

In [ ]:
from processing import predict_with_metadata
from nbutils import display_polars

processing_result = genre

predictions = predict_with_metadata(
    processing_result=processing_result, 
    model=models, 
    songs_df=songs_df
)

display_polars(predictions)

shape: (50, 5)
┌───────────┬─────────────────────┬─────────────────────┬─────────────────────┬────────────────────┐
│ song_id   ┆ artist_name         ┆ song_title          ┆ linear_predicted_ta ┆ random_forest_pred │
│ ---       ┆ ---                 ┆ ---                 ┆ gs                  ┆ icted_tags         │
│ str       ┆ str                 ┆ str                 ┆ ---                 ┆ ---                │
│           ┆                     ┆                     ┆ list[str]           ┆ list[str]          │
╞═══════════╪═════════════════════╪═════════════════════╪═════════════════════╪════════════════════╡
│ 79369571  ┆ Bill Withers        ┆ Lovely Day          ┆ ["Hip-Hop", "Pop",  ┆ ["Funk", "Soul"]   │
│           ┆                     ┆                     ┆ "Soul"]             ┆                    │
│ 135893281 ┆ Mirage              ┆ Summer Grooves      ┆ ["Disco", "Funk",   ┆ ["Funk", "Soul"]   │
│           ┆                     ┆                     ┆ "Hip-Hop", "Soul"]

In [91]:
# Explode the list column and get unique values
unique_predicted_tags = (
    predictions
    .select("predicted_tags")
    .explode("predicted_tags")
    .unique()
    .sort("predicted_tags")
)

print("Unique predicted tags:")
print(unique_predicted_tags)

Unique predicted tags:
shape: (15, 1)
┌────────────────┐
│ predicted_tags │
│ ---            │
│ str            │
╞════════════════╡
│ null           │
│ Ambient        │
│ Bass           │
│ Beats          │
│ Breakbeat      │
│ Disco          │
│ Funk           │
│ Garage         │
│ House          │
│ Indie          │
│ Pop            │
│ Rap            │
│ Soul           │
│ Techno         │
│ Trance         │
└────────────────┘
